In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# --- Data Loading and Cleaning ---
df = pd.read_csv('dataset_mansion.csv', index_col=0)

df['Consumption'] = pd.to_numeric(
    df['Consumption'].astype('string').str.replace(',', '.', regex=False),
    errors='raise'
)

df['MTime'] = pd.to_datetime(df['MTime'], errors='raise', utc=True)
assert df['MTime'].notna().all(), 'Check missing timestamps first.'
assert not df['MTime'].duplicated().any(), 'Check duplicate timestamps first.'
df = df.set_index('MTime').sort_index()

# --- Feature Removal ---
columns_to_drop = [
    'Latitude', 'Longitude', 'Place', 'PresentWeather(rank)',
    'HighestTemperature(degC)', 'LowestTemperature(degC)',
    'MaximumWindSpeed(m/s)', 'MinimumWindSpeed(m/s)',
    'MaximumPrecipitationIntensity(mm/h)'
]
df = df.drop(columns=columns_to_drop)

# --- Seasonal Imputation ---
df = df.asfreq('h')

# 1. Safely interpolate ONLY small gaps (e.g., up to 4 hours)
df = df.interpolate(method='time', limit=4)
# 2. For the 20 to 58-hour gaps, fill using the exact same hour the day before (up to 72 hrs)
for _ in range(3): 
    df = df.fillna(df.shift(24))
# 3. If any gaps survive, fill with the historical average for that specific month and hour
df = df.fillna(df.groupby([df.index.month, df.index.hour]).transform('mean'))
# 4. Final safety net
df = df.bfill().ffill()

# --- Feature Engineering ---
df_features = df.copy()
local_time = df_features.index.tz_convert('Europe/Helsinki')
hour = local_time.hour
month = local_time.month - 1
day_of_week = local_time.dayofweek

df_features['hour_sin'] = np.sin(2 * np.pi * hour / 24)
df_features['hour_cos'] = np.cos(2 * np.pi * hour / 24)
df_features['day_sin'] = np.sin(2 * np.pi * day_of_week / 7)
df_features['day_cos'] = np.cos(2 * np.pi * day_of_week / 7)
df_features['month_sin'] = np.sin(2 * np.pi * month / 12)
df_features['month_cos'] = np.cos(2 * np.pi * month / 12)

# --- Wind Direction Transformation ---
# 1. Calculate the cyclical nature of the wind
wind_radians = df_features['WindDirection(deg)'] * (np.pi / 180)
df_features['wind_dir_sin'] = np.sin(wind_radians)
df_features['wind_dir_cos'] = np.cos(wind_radians)

# ßDROP the raw wind direction column now that we have the clean sin/cos versions
df_features = df_features.drop(columns=['WindDirection(deg)'])


df_features['is_weekend'] = (local_time.dayofweek >= 5).astype(int)
df_features['is_covid'] = (local_time >= '2020-03-01').astype(int)

# Holidays
week_of_year = local_time.isocalendar().week
local_month = local_time.month
day = local_time.day
is_summer_holiday = ((local_month == 6) & (day >= 8)) | (local_month == 7) | ((local_month == 8) & (day <= 7))
df_features['is_holiday'] = (is_summer_holiday | (week_of_year == 9) | (week_of_year == 42) | ((local_month == 12) & (day >= 24) & (day <= 26))).astype(int)

# Working hours & routines
df_features['is_working_hour'] = ((hour >= 8) & (hour < 16) & (df_features['is_weekend'] == 0) & (df_features['is_holiday'] == 0)).astype(int)
df_features['is_evening_routine'] = ((hour >= 17) & (hour <= 23)).astype(int)

# --- Splitting & Scaling ---
train_df = df_features.loc['2016-01-01':'2020-09-30'].copy()
test_df = df_features.loc['2020-10-01':'2021-09-30'].copy()
eval1_df = df_features.loc['2021-10-01':'2021-12-31'].copy()
eval2_df = df_features.loc['2022-01-01':'2022-02-28'].copy()
eval3_df = df_features.loc['2022-03-01':'2022-05-31'].copy()

# --- 1. UPDATE SCALING (In Cell 1: Include 'Consumption' in StandardScaler) ---
continuous_features = [
    'Consumption', 'AirTemperature(degC)', 'RelativeHumidity(%)', 
    'WindSpeed(m/s)', 'PrecipitationAmount(mm)', 'AirPressure(hPa)'
]
scaler = StandardScaler()

train_df[continuous_features] = scaler.fit_transform(train_df[continuous_features])
test_df[continuous_features] = scaler.transform(test_df[continuous_features])
eval1_df[continuous_features] = scaler.transform(eval1_df[continuous_features])
eval2_df[continuous_features] = scaler.transform(eval2_df[continuous_features])
eval3_df[continuous_features] = scaler.transform(eval3_df[continuous_features])

# Save Consumption mean & std to convert predictions back to real kWh later
cons_mean = scaler.mean_[0]
cons_std = scaler.scale_[0]

print("Preprocessing complete. Data is ready for the TCN.")

Preprocessing complete. Data is ready for the TCN.
